In [ ]:
#====================================================================================================#
#                                                                                                    #
#                                                        ██╗   ██╗   ████████╗ █████╗ ██████╗        #
#      AEC2 - BAIN                                       ██║   ██║   ╚══██╔══╝██╔══██╗██╔══██╗       #
#                                                        ██║   ██║█████╗██║   ███████║██║  ██║       #
#      created:        08/04/2026  -  01:00:51           ██║   ██║╚════╝██║   ██╔══██║██║  ██║       #
#      last change:    23/04/2026  -  12:14:43           ╚██████╔╝      ██║   ██║  ██║██████╔╝       #
#                                                         ╚═════╝       ╚═╝   ╚═╝  ╚═╝╚═════╝        #
#                                                                                                    #
#      Ismael Hernandez Clemente                         ismael.hernandez@live.u-tad.com             #
#                                                                                                    #
#      Github:                                           https://github.com/ismaelucky342            #
#                                                                                                    #
#====================================================================================================# 

# AEC2 – Análisis Avanzado de Sentimiento y Tendencias en Twitter

En esta entrega amplío la clase `DataExtractor` de la AEC1 con cuatro bloques nuevos:

1. **Conexión a la API de Twitter** vía RapidAPI (`load_data_api`)
2. **Modelado de tópicos** con LDA y gensim (`model_topics`)
3. **Análisis de sentimiento** con TextBlob y spaCy (`analyze_sentiment`)
4. **Parsing y resumen extractivo** con NLTK (`parse_and_summarize`)

Como dataset de trabajo uso el de tweets de Bitcoin de Kaggle, que carga bien con la clase de la AEC1. Si la API devuelve pocos resultados (suele pasar con el tier gratuito), cambio al CSV local sin tocar nada más.

In [ ]:
import os
import re
import unicodedata
from pathlib import Path

import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud

print("Librerías base cargadas.")

---
## Clase `DataExtractor` (AEC2)

Parto de la clase de la AEC1 y añado los métodos nuevos. Todo sigue centralizado aquí para que sea fácil de reutilizar desde scripts o desde el dashboard.

In [ ]:
# Importo la clase centralizada desde data_extractor.py
from data_extractor import DataExtractor

print("DataExtractor importada correctamente desde data_extractor.py")

---
## 1. Extracción de datos vía API

Me conecto a la API de Twitter a través de RapidAPI usando `requests`. La clave la tengo en la variable de entorno `RAPIDAPI_KEY` para no exponerla aquí. Si la API no devuelve suficientes tweets (pasa mucho con el tier gratuito), el bloque siguiente cae al CSV local de Kaggle.

In [ ]:
# Compruebo que la clave está configurada antes de lanzar la petición.
api_key = os.environ.get("RAPIDAPI_KEY", "")
if not api_key:
    print("  RAPIDAPI_KEY no encontrada. Exporta la variable antes de ejecutar:")
    print("   export RAPIDAPI_KEY='tu_clave_aqui'")
else:
    print(f" RAPIDAPI_KEY encontrada ({len(api_key)} caracteres)")

In [ ]:
# Intento extraer de la API. Si falla o devuelve poco, uso el CSV local.
extractor = DataExtractor()

try:
    df_api = extractor.load_data_api(query="#bitcoin", max_results=100, output_file="tweets_api.csv")
    print(df_api[['user_name', 'date', 'text']].head(3))
    USE_API_DATA = True
except Exception as e:
    print(f"[API] No disponible: {e}")
    print("→ Cambiando al dataset local CSV.")
    USE_API_DATA = False

---
## 2. Carga del dataset local (fallback o dataset principal)

Si la API no devuelve suficiente volumen, cargo el dataset de Bitcoin de Kaggle. Con este sí tengo los resultados representativos para el análisis.

In [ ]:
SOURCE_FILE = 'Bitcoin_tweets_dataset_2.csv'

if not USE_API_DATA:
    extractor = DataExtractor(source=SOURCE_FILE, chunksize=100_000)
    df = extractor.load_data()
else:
    # Si la API funcionó, uso esos datos para el resto del notebook.
    df = extractor.data

print(f"\nShape      : {df.shape}")
print(f"Columnas   : {list(df.columns)}")
print(f"Memoria    : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head(3)

In [ ]:
# Chequeo rápido de nulos.
print("Nulos por columna:")
print(df.isnull().sum())

---
## 3. Limpieza y extracción de hashtags

Aplico `clean_text` para normalizar el texto y luego `extract_hashtags` para sacar los hashtags. Comparo un tweet en bruto con su versión limpia para ver el efecto del preprocesamiento.

In [ ]:
# Comparo bruto vs limpio en un ejemplo.
sample_raw = df['text'].dropna().iloc[0]
print("BRUTO :", repr(sample_raw))
print("LIMPIO:", repr(extractor.clean_text(sample_raw)))

In [ ]:
# Aplico la limpieza a todo el dataset.
df['clean_text'] = df['text'].apply(extractor.clean_text)
df['hashtags']   = df['clean_text'].apply(extractor.extract_hashtags)

n_con_hashtags = (df['hashtags'].apply(len) > 0).sum()
print(f"Tweets con al menos un hashtag: {n_con_hashtags:,} / {len(df):,} ({n_con_hashtags/len(df)*100:.1f}%)")

pd.set_option('display.max_colwidth', 100)
df[['text', 'hashtags']].head(5)

---
## 4. Modelado de tópicos con LDA

Aplico Latent Dirichlet Allocation con gensim para descubrir los temas principales del corpus. LDA asume que cada documento es una mezcla de tópicos, y cada tópico es una distribución de palabras. Con 5 tópicos y 10 palabras por tópico puedo hacerme una idea de qué se habla en los tweets de Bitcoin.

In [ ]:
# Para el LDA trabajo sobre una muestra si el dataset es muy grande,
# ya que LDA es costoso computacionalmente.
N_SAMPLE = min(50_000, len(df))
extractor.data = df.sample(N_SAMPLE, random_state=42).reset_index(drop=True)

print(f"Entrenando LDA sobre {N_SAMPLE:,} tweets...")
topics = extractor.model_topics(num_topics=5, passes=10)
print("\nTópicos descubiertos:")

In [ ]:
# Muestro los tópicos con un formato limpio.
for i, topic in enumerate(topics):
    print(f"  Tópico {i+1}: {', '.join(topic)}")

In [ ]:
# Visualizo los tópicos como un heatmap de palabras clave.
import numpy as np

# Reconstruyo la matriz topic x word para el heatmap.
all_words = list(dict.fromkeys(w for t in topics for w in t))  # palabras únicas en orden
matrix = np.zeros((len(topics), len(all_words)))
for i, topic in enumerate(topics):
    for j, word in enumerate(all_words):
        if word in topic:
            matrix[i][j] = topic.index(word) + 1  # peso inverso al rango

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(all_words)))
ax.set_xticklabels(all_words, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(topics)))
ax.set_yticklabels([f'Tópico {i+1}' for i in range(len(topics))])
ax.set_title('Palabras clave por tópico (LDA)', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Relevancia')
plt.tight_layout()
plt.savefig('lda_topics_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Heatmap guardado como 'lda_topics_heatmap.png'")

Cada fila es un tópico y cada columna una palabra clave. Las celdas más oscuras indican mayor relevancia de esa palabra en ese tópico. Se pueden ver clusters temáticos claros: precios y especulación, comunidad/hodl, noticias y adopción institucional, minería y tecnología, y trading.

---
## 5. Análisis de sentimiento

Calculo la polaridad y subjetividad de cada tweet con TextBlob. La polaridad va de -1 (muy negativo) a +1 (muy positivo), y la subjetividad de 0 (objetivo) a 1 (subjetivo). También pruebo spaCy con spacytextblob como alternativa más precisa.

In [ ]:
# Restauro el dataset completo antes del análisis de sentimiento.
extractor.data = df.copy()

print("Analizando sentimiento con TextBlob...")
df_sent = extractor.analyze_sentiment(method='textblob')
print("Hecho.")
df_sent[['clean_text', 'sentiment_polarity', 'sentiment_subjectivity']].head(5)

In [ ]:
# Estadísticas básicas de sentimiento.
print("Polaridad:")
print(df_sent['sentiment_polarity'].describe().round(4))
print("\nSubjetividad:")
print(df_sent['sentiment_subjectivity'].describe().round(4))

In [ ]:
# Clasifico los tweets por sentimiento para facilitar la visualización.
def classify_sentiment(pol):
    if pol > 0.05:   return 'Positivo'
    elif pol < -0.05: return 'Negativo'
    else:             return 'Neutro'

df_sent['sentiment_label'] = df_sent['sentiment_polarity'].apply(classify_sentiment)

counts = df_sent['sentiment_label'].value_counts()
pcts   = (counts / len(df_sent) * 100).round(1)
print("Distribución de sentimientos:")
for label in ['Positivo', 'Neutro', 'Negativo']:
    print(f"  {label}: {counts.get(label, 0):,} tweets ({pcts.get(label, 0)}%)")

In [ ]:
# Visualizo la distribución de sentimentos.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Tarta de distribución.
colors_pie = ['#2ecc71', '#95a5a6', '#e74c3c']
labels_ord  = ['Positivo', 'Neutro', 'Negativo']
sizes       = [counts.get(l, 0) for l in labels_ord]
axes[0].pie(sizes, labels=labels_ord, autopct='%1.1f%%', colors=colors_pie, startangle=140)
axes[0].set_title('Distribución de sentimiento', fontweight='bold')

# 2. Histograma de polaridad.
axes[1].hist(df_sent['sentiment_polarity'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1, label='Neutral')
axes[1].set_xlabel('Polaridad')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de polaridad', fontweight='bold')
axes[1].legend()

# 3. Scatter polaridad vs subjetividad.
color_map = {'Positivo': '#2ecc71', 'Neutro': '#95a5a6', 'Negativo': '#e74c3c'}
sample_scatter = df_sent.sample(min(3000, len(df_sent)), random_state=42)
for label, color in color_map.items():
    subset = sample_scatter[sample_scatter['sentiment_label'] == label]
    axes[2].scatter(subset['sentiment_polarity'], subset['sentiment_subjectivity'],
                    alpha=0.3, s=5, color=color, label=label)
axes[2].set_xlabel('Polaridad')
axes[2].set_ylabel('Subjetividad')
axes[2].set_title('Polaridad vs Subjetividad', fontweight='bold')
axes[2].legend(markerscale=3)

plt.tight_layout()
plt.savefig('sentiment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado como 'sentiment_analysis.png'")

In [ ]:
# Muestro ejemplos representativos de cada categoría.
print("=== Ejemplos de tweets POSITIVOS ===")
pos_examples = df_sent[df_sent['sentiment_label'] == 'Positivo'].nlargest(3, 'sentiment_polarity')
for _, row in pos_examples.iterrows():
    print(f"  [pol={row['sentiment_polarity']:.2f}] {row['text'][:120]}")

print("\n=== Ejemplos de tweets NEGATIVOS ===")
neg_examples = df_sent[df_sent['sentiment_label'] == 'Negativo'].nsmallest(3, 'sentiment_polarity')
for _, row in neg_examples.iterrows():
    print(f"  [pol={row['sentiment_polarity']:.2f}] {row['text'][:120]}")

---
## 6. Evolución temporal del sentimiento

Cruzo el análisis de sentimiento con la dimensión temporal para ver si hay períodos de euforia o pánico en la comunidad Bitcoin.

In [ ]:
# Convierto la fecha y calculo la polaridad media diaria.
df_sent['date'] = pd.to_datetime(df_sent['date'], utc=True, errors='coerce').dt.date
daily_sentiment = (
    df_sent.groupby('date')['sentiment_polarity']
           .agg(['mean', 'std', 'count'])
           .reset_index()
           .rename(columns={'mean': 'pol_mean', 'std': 'pol_std', 'count': 'n_tweets'})
)
daily_sentiment = daily_sentiment[daily_sentiment['n_tweets'] >= 20]  # Filtro días con pocos tweets.

fig, ax = plt.subplots(figsize=(14, 5))
dates = pd.to_datetime(daily_sentiment['date'])
ax.plot(dates, daily_sentiment['pol_mean'], color='steelblue', linewidth=1.5, label='Polaridad media')
ax.fill_between(dates,
                daily_sentiment['pol_mean'] - daily_sentiment['pol_std'].fillna(0),
                daily_sentiment['pol_mean'] + daily_sentiment['pol_std'].fillna(0),
                alpha=0.15, color='steelblue', label='±1 std')
ax.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Fecha')
ax.set_ylabel('Polaridad media')
ax.set_title('Evolución temporal del sentimiento en tweets de Bitcoin', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('sentiment_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Guardado como 'sentiment_evolution.png'")

---
## 7. Parsing y resumen extractivo

Con NLTK tokenizo el corpus completo en oraciones, calculo la frecuencia de palabras clave (sin stopwords) y selecciono las oraciones más representativas. Es un enfoque extractivo puro: no genero texto nuevo, selecciono el que ya existe.

In [ ]:
# Trabajo sobre una muestra para que el resumen sea ágil.
SAMPLE_SUMMARY = min(10_000, len(df))
extractor.data = df.sample(SAMPLE_SUMMARY, random_state=42).reset_index(drop=True)

print(f"Generando resumen extractivo sobre {SAMPLE_SUMMARY:,} tweets...")
summary = extractor.parse_and_summarize(summary_ratio=0.01)  # 1% de las oraciones

print(f"\nResumen ({len(summary.split())} palabras):")
print("-" * 80)
print(summary[:1500])
print("..." if len(summary) > 1500 else "")

In [ ]:
# Visualizo las palabras más frecuentes del corpus (sin stopwords ni hashtags).
from collections import Counter
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

word_counts = Counter()
for tweet in df['clean_text'].dropna():
    words = [w for w in tweet.split()
             if not w.startswith('#') and w not in stop_words and len(w) > 2]
    word_counts.update(words)

top_words = word_counts.most_common(20)
words_df  = pd.DataFrame(top_words, columns=['word', 'count'])

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(words_df['word'][::-1], words_df['count'][::-1], color='darkorange')
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_xlabel('Frecuencia')
ax.set_title('Top 20 palabras más frecuentes (sin hashtags ni stopwords)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('top_words.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Árbol sintáctico (parsing con spaCy)

Uso spaCy para parsear la estructura sintáctica de algunos tweets. Visualizo el árbol de dependencias de un par de ejemplos representativos.

In [ ]:
import spacy
from spacy import displacy

# Cargo el modelo en inglés.
try:
    nlp = spacy.load('en_core_web_sm')
    print("Modelo spaCy cargado.")
except OSError:
    print("Descargando modelo spaCy... ejecuta: python -m spacy download en_core_web_sm")
    nlp = None

In [ ]:
if nlp is not None:
    # Selecciono tweets que no sean demasiado cortos para que el árbol sea interesante.
    sample_tweets = (
        df[df['clean_text'].str.split().apply(len) > 8]
        ['clean_text'].dropna().head(3).tolist()
    )

    for i, tweet in enumerate(sample_tweets, 1):
        doc = nlp(tweet[:200])  # Limito a 200 chars para que displacy no explote.
        print(f"\n--- Tweet {i} ---")
        print(tweet[:200])
        print("\nTokens y dependencias:")
        for token in doc:
            print(f"  {token.text:<20} POS: {token.pos_:<10} DEP: {token.dep_:<15} → {token.head.text}")

In [ ]:
if nlp is not None:
    # Renderizo el árbol del primer tweet en SVG para guardarlo.
    doc_sample = nlp(sample_tweets[0][:200])
    svg = displacy.render(doc_sample, style='dep', jupyter=False)
    with open('dependency_tree.svg', 'w', encoding='utf-8') as f:
        f.write(svg)
    print("Árbol de dependencias guardado como 'dependency_tree.svg'")

    # También lo muestro inline si estamos en Jupyter.
    displacy.render(doc_sample, style='dep', jupyter=True)

---
## 10. Exportación de resultados

Guardo todos los outputs intermedios y finales como archivos CSV y TXT.
Así cualquiera puede revisar o reproducir los resultados sin ejecutar el notebook entero.


In [ ]:
# Exporto todos los resultados intermedios y finales a la carpeta output/.
# Esto cubre el dataset limpio, las tablas de hashtags, el sentimiento y el resumen.
extractor.data = df_sent.copy()  # Me aseguro de que data tiene las columnas de sentimiento.
extractor.export_results(output_dir="output")

# Exporto los tópicos LDA por separado.
extractor.export_topics(topics, output_dir="output")

print("
Archivos generados en output/:")
import os
for f in sorted(os.listdir("output")):
    size = os.path.getsize(f"output/{f}")
    print(f"  {f:<35} {size:>10,} bytes")


In [ ]:
extractor.data = df.copy()
analytics = extractor.analytics_hashtags_extended()
overall   = analytics['overall']

# Recomputo las etiquetas de sentimiento sobre el df completo.
if 'sentiment_label' not in df.columns:
    df = extractor.analyze_sentiment(method='textblob')
    df['sentiment_label'] = df['sentiment_polarity'].apply(classify_sentiment)

sent_dist = df['sentiment_label'].value_counts()

print("=" * 60)
print("RESUMEN DEL ANÁLISIS – AEC2")
print("=" * 60)
print(f"  Total de tweets analizados : {len(df):,}")
print(f"  Hashtags únicos            : {len(overall):,}")
print(f"  Usuarios únicos            : {df['user_name'].nunique():,}")
print(f"\n  Sentimiento (TextBlob):")
for label in ['Positivo', 'Neutro', 'Negativo']:
    n = sent_dist.get(label, 0)
    print(f"    {label:<12}: {n:>8,} ({n/len(df)*100:.1f}%)")
print(f"\n  Top 5 hashtags:")
for _, row in overall.head(5).iterrows():
    print(f"    #{row['hashtag']:<20} {row['frequency']:>8,} usos")
print(f"\n  Tópicos LDA descubiertos:")
for i, t in enumerate(topics, 1):
    print(f"    Tópico {i}: {', '.join(t[:5])}...")
print("=" * 60)

In [ ]:
# Genero la wordcloud final para cerrar el notebook visualmente.
extractor.generate_hashtag_wordcloud(overall_df=overall, max_words=100, figsize=(14, 7))

---
## UNIDAD 6 – Análisis de Redes e Integración de LLMs

Esta sección extiende la clase `DataExtractor` con tres bloques nuevos:
1. **Análisis de Redes con NetworkX** – grafo de menciones, métricas de centralidad y comunidades.
2. **Generación de Prompt** – resumen de insights de la red.
3. **Chat con LLM** – modelo local Gemma-2-2B-it o API Gemini como fallback.


### 6.1 Construir el grafo de interacciones

El método `build_interaction_graph()` construye un **dígrafo ponderado** donde cada nodo es un usuario y cada arista A→B indica que A mencionó a B en un tweet. El peso refleja la frecuencia de la interacción.


In [ ]:
# CELDA U6-1  Instalar dependencias extra (solo la primera vez)
# !pip install networkx python-louvain transformers torch accelerate --quiet

# Recarga el módulo para incluir los nuevos métodos
import importlib, sys
if 'data_extractor' in sys.modules:
    importlib.reload(sys.modules['data_extractor'])
from data_extractor import DataExtractor


In [ ]:
# CELDA U6-2  Construir grafo
# Asumimos que `ext` ya tiene datos cargados; si no:
# ext = DataExtractor(source='Bitcoin_tweets_dataset_2.csv')
# ext.load_data()

G = ext.build_interaction_graph()
print(f'Nodes: {G.number_of_nodes():,}  |  Edges: {G.number_of_edges():,}')


### 6.2 Métricas de red y visualización

| Métrica | Interpretación |
|---|---|
| **In-degree centrality** | Usuarios más mencionados |
| **Out-degree centrality** | Usuarios que más mencionan a otros |
| **Betweenness centrality** | Nodos puente entre comunidades |
| **PageRank** | Influencia propagada por la red |

Detección de comunidades con algoritmo de **Louvain** (fallback: greedy modularity).


In [ ]:
# CELDA U6-3  Analizar la red
metrics = ext.analyze_network(G)

print('\nTop-3 by PageRank:')
for node, val in metrics['top_nodes']['pagerank']:
    print(f'  @{node}  PR={val:.4f}')

print(f'\nCommunities detected: {metrics["n_communities"]}')


### 6.3 Generar el prompt para el LLM

`generate_prompt_from_network()` extrae los insights más relevantes y construye un prompt que pide al LLM interpretación cualitativa de la red.


In [ ]:
# CELDA U6-4  Generar prompt
prompt = ext.generate_prompt_from_network(G)


### 6.4 Chat con LLM local (Gemma-2-2B-it)

`chat_local_llm()` carga el modelo y abre una sesión de chat multi-turno. Escribe `exit` para salir.

**Requisitos:** GPU con ≥6 GB VRAM, o Google Colab T4 gratuita.

> Si no tienes GPU suficiente usa la celda U6-6 (API Gemini).


In [ ]:
# CELDA U6-5  Chat local
# Modelos alternativos más ligeros:
#   model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'   # ~2.2 GB RAM
#   model_name = 'microsoft/phi-2'                       # ~5.5 GB

ext.chat_local_llm(
    prompt=prompt,
    model_name='google/gemma-2-2b-it',
    max_new_tokens=512,
    temperature=0.7,
)


### 6.5 Alternativa: API de Gemini (sin GPU)

Obtén clave gratuita en https://aistudio.google.com/app/apikey (15 req/min).


In [ ]:
# CELDA U6-6  Fallback Gemini API
import os
# os.environ['GEMINI_API_KEY'] = 'TU_CLAVE_AQUI'

response = ext.api_llm_fallback(
    prompt=prompt,
    model='gemini-1.5-flash',
    api_key_env='GEMINI_API_KEY',
)
print(response)


### 6.6 Pipeline completo U6

```
load_data()  →  build_interaction_graph()  →  analyze_network()
             →  generate_prompt_from_network()  →  chat_local_llm() / api_llm_fallback()
```


In [ ]:
# CELDA U6-7  Pipeline completo
from data_extractor import DataExtractor

# 1. Datos
ext = DataExtractor(source='Bitcoin_tweets_dataset_2.csv')
ext.load_data()

# 2. Grafo
G = ext.build_interaction_graph()

# 3. Análisis
metrics = ext.analyze_network(G)

# 4. Prompt
prompt = ext.generate_prompt_from_network(G)

# 5a. LLM local
# ext.chat_local_llm(prompt=prompt)

# 5b. API Gemini (sin GPU)
# ext.api_llm_fallback(prompt=prompt)
